# Player Style Clustering

This notebook presents an unsupervised ML case study over player behavior profiles from KnightVision `silver_games`. The production implementation lives in `ml.player_style_clustering`; this notebook reads the generated artifacts and explains the result.

The output labels are statistical personas, not ground-truth identities.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Image, Markdown, display
except ImportError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename: str):
            self.filename = filename

        def __repr__(self) -> str:
            return f"Image({self.filename})"

    def display(value) -> None:
        print(value)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "models").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def require_artifacts(paths: list[Path]) -> None:
    missing = [path for path in paths if not path.exists()]
    if missing:
        display(Markdown("### Missing artifacts"))
        display(Markdown("Run the matching `make train-*` target, then rerun this notebook."))
        for path in missing:
            display(Markdown(f"- `{path.relative_to(REPO_ROOT)}`"))
        raise FileNotFoundError("Required ML artifacts are missing")


def metric_frame(metrics: dict[str, float]) -> pd.DataFrame:
    return (
        pd.Series(metrics, name="value")
        .rename_axis("metric")
        .reset_index()
        .assign(value=lambda frame: frame["value"].map(lambda value: f"{float(value):.4f}"))
    )


def display_markdown_file(path: Path) -> None:
    display(Markdown(path.read_text(encoding="utf-8")))


print(f"Repo root: {REPO_ROOT}")

ARTIFACT_DIR = REPO_ROOT / "models" / "player_style_clusters"
required = [
    ARTIFACT_DIR / "metrics.json",
    ARTIFACT_DIR / "cluster_profiles.csv",
    ARTIFACT_DIR / "cluster_sweep.csv",
    ARTIFACT_DIR / "cluster_scatter.png",
    ARTIFACT_DIR / "feature_profiles.png",
    ARTIFACT_DIR / "elbow_plot.png",
    ARTIFACT_DIR / "silhouette_by_k.png",
    ARTIFACT_DIR / "evaluation_report.md",
    ARTIFACT_DIR / "model_card.md",
]
require_artifacts(required)
metadata = read_json(ARTIFACT_DIR / "metrics.json")
metadata.keys()


## Dataset And Objective

Each row represents one eligible player aggregated from Silver games. The clustering model groups players by behavior features such as win/loss/draw profile, game length, capture behavior, castling rate, opening diversity, and time-control preferences.

Because this is unsupervised, there is no accuracy score. Evaluation focuses on whether the clusters are stable enough to describe and whether their feature profiles are meaningfully different.


In [ ]:
metrics = metadata["metrics"]
pd.DataFrame(
    [
        ("Eligible players", metrics["eligible_players"]),
        ("Clusters", metrics["clusters"]),
        ("Inertia", metrics["inertia"]),
        ("Silhouette score", metrics["silhouette_score"]),
        ("PCA explained variance ratio", metrics["pca_explained_variance_ratio"]),
    ],
    columns=["item", "value"],
)


## Feature Set

The feature set intentionally focuses on behavior rather than identity. Player names are not used as model features.


In [ ]:
pd.DataFrame({"feature": metadata["features"]})


## Cluster Count Sweep

The training script compares multiple values of `k` using inertia and silhouette score. The selected model uses five clusters so the personas remain readable while preserving meaningful separation.


In [ ]:
sweep = pd.read_csv(ARTIFACT_DIR / "cluster_sweep.csv")
sweep


In [ ]:
display(Image(filename=str(ARTIFACT_DIR / "elbow_plot.png")))
display(Image(filename=str(ARTIFACT_DIR / "silhouette_by_k.png")))


## Cluster Profiles

Cluster labels are generated from the feature profile of each cluster. They are useful labels for communication, not labels provided by Lichess or by human annotation.


In [ ]:
profiles = pd.read_csv(ARTIFACT_DIR / "cluster_profiles.csv")
profiles


In [ ]:
display(Image(filename=str(ARTIFACT_DIR / "feature_profiles.png")))


## 2D Projection

The scatter plot uses PCA only for visualization. The KMeans model is trained on the scaled feature matrix, not on the two-dimensional plot alone.


In [ ]:
display(Image(filename=str(ARTIFACT_DIR / "cluster_scatter.png")))


## Player Assignment Privacy

`cluster_assignments.csv` is generated locally, but it is intentionally ignored by git because it includes public player identifiers. This notebook does not display individual player assignments by default.


In [ ]:
assignment_path = ARTIFACT_DIR / "cluster_assignments.csv"
if assignment_path.exists():
    assignment_columns = pd.read_csv(assignment_path, nrows=0).columns.tolist()
    print(f"Local assignment file exists with {len(assignment_columns)} columns, but player rows are not displayed by default.")
else:
    print("No local cluster assignment file found. This is expected on a clean clone.")


## Report And Model Card

These generated artifacts are the durable evaluation summary for the clustering case study.


In [ ]:
display_markdown_file(ARTIFACT_DIR / "evaluation_report.md")


In [ ]:
display_markdown_file(ARTIFACT_DIR / "model_card.md")


## Interpretation

The silhouette score is modest, which is normal for noisy human behavior data. The value of this case study is not perfect segmentation; it is showing a reproducible unsupervised workflow that turns player-level aggregates into interpretable chess-style personas.


## Optional Retraining

Set `RUN_TRAINING = True` only when the benchmark DuckDB warehouse exists and you want to regenerate artifacts.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ml.player_style_clustering.train import TrainingConfig, train

    metadata = train(
        TrainingConfig(
            duckdb_path=REPO_ROOT / "warehouse" / "knightvision_benchmark.duckdb",
            output_dir=ARTIFACT_DIR,
            min_games=10,
            clusters=5,
        )
    )
    metadata["metrics"]
else:
    print("Skipping training. Set RUN_TRAINING = True to regenerate artifacts.")
